# 01 — Embeddings: From Zero to Semantic Search
### Cortex-RAG Series | github.com/ather-ops
---
**What you will learn in this notebook:**
- What is an embedding — the math behind it
- One-hot encoding vs dense vectors
- Embedding matrix multiplication from scratch
- Cosine similarity — measure meaning not just words
- SentenceTransformer — real pretrained embeddings
- Semantic search on real data
- Save embeddings to CSV for later use in RAG

> This notebook is the **backbone** of every RAG system. Master this first.

## What is an Embedding?

An embedding converts text into a list of numbers.

These numbers **capture meaning** — not just spelling.

| Word | Normal representation | Embedding |
|------|----------------------|-----------|
| "King" | Random index | [0.2, 0.8, 0.1, ...] |
| "Queen" | Random index | [0.2, 0.7, 0.2, ...] |
| "Pizza" | Random index | [0.9, 0.1, 0.8, ...] |

King and Queen have **similar numbers** because they have similar meaning.
King and Pizza have **different numbers** because they mean different things.

**This is how machines understand language.**

## Step 1 — Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("All libraries imported successfully")

All libraries imported successfully


## Step 2 — Level 0: One-Hot Encoding (The Old Way)

Before embeddings, computers used one-hot encoding.
Every word got a vector with all zeros except one position.

**Problem:** No meaning captured. King and Queen look completely unrelated.

In [2]:
print("="*50)
print("LEVEL 0 — One-Hot Encoding")
print("="*50)

vocabulary = ["king", "queen", "pizza", "burger"]
word_to_index = {w:i for i,w in enumerate(vocabulary)}

def one_hot(word, vocab):
    vec = np.zeros(len(vocab))
    vec[vocab[word]] = 1
    return vec

king_oh   = one_hot("king",   word_to_index)
queen_oh  = one_hot("queen",  word_to_index)
pizza_oh  = one_hot("pizza",  word_to_index)

print(f"king  : {king_oh}")
print(f"queen : {queen_oh}")
print(f"pizza : {pizza_oh}")

sim_king_queen = np.dot(king_oh, queen_oh)
sim_king_pizza = np.dot(king_oh, pizza_oh)

print(f"\nSimilarity king-queen : {sim_king_queen}")
print(f"Similarity king-pizza : {sim_king_pizza}")
print("Problem: Both show 0 — one-hot cannot measure meaning")

LEVEL 0 — One-Hot Encoding
king  : [1. 0. 0. 0.]
queen : [0. 1. 0. 0.]
pizza : [0. 0. 1. 0.]

Similarity king-queen : 0.0
Similarity king-pizza : 0.0
Problem: Both show 0 — one-hot cannot measure meaning


## Step 3 — Level 1: Manual Embedding Matrix

We fix the one-hot problem by multiplying with an **embedding matrix**.

The embedding matrix maps each word to a dense vector with actual values.
These values are learned during training — but here we set them manually to understand the concept.

In [3]:
print("="*50)
print("LEVEL 1 — Manual Embedding Matrix")
print("="*50)

E = np.array([
    [0.8, 0.9, 0.1],
    [0.8, 0.8, 0.1],
    [0.1, 0.1, 0.9],
    [0.1, 0.2, 0.8],
])

print("Embedding matrix shape:", E.shape)
print("Rows = words, Columns = learned features")
print()

king_emb   = np.dot(one_hot("king", word_to_index), E)
queen_emb  = np.dot(one_hot("queen", word_to_index), E)
pizza_emb  = np.dot(one_hot("pizza", word_to_index), E)

print(f"king  embedding: {king_emb}")
print(f"queen embedding: {queen_emb}")
print(f"pizza embedding: {pizza_emb}")

LEVEL 1 — Manual Embedding Matrix
Embedding matrix shape: (4, 3)
Rows = words, Columns = learned features

king  embedding: [0.8 0.9 0.1]
queen embedding: [0.8 0.8 0.1]
pizza embedding: [0.1 0.1 0.9]


## Step 4 — Cosine Similarity from Scratch

In [4]:
print("="*50)
print("COSINE SIMILARITY — From Scratch")
print("="*50)

def cosine_sim(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    return dot / (norm1 * norm2)

sim_kq = cosine_sim(king_emb, queen_emb)
sim_kp = cosine_sim(king_emb, pizza_emb)

print(f"Cosine similarity king <-> queen : {sim_kq:.4f} (HIGH)")
print(f"Cosine similarity king <-> pizza : {sim_kp:.4f} (LOW)")
print()

sim_matrix = np.array([[cosine_sim(E[i], E[j]) for j in range(4)] for i in range(4)])
print("Similarity matrix (all pairs):")
print(np.round(sim_matrix, 3))

COSINE SIMILARITY — From Scratch
Cosine similarity king <-> queen : 0.9981 (HIGH)
Cosine similarity king <-> pizza : 0.2712 (LOW)

Similarity matrix (all pairs):
[[1.    0.998 0.271 0.279]
 [0.998 1.    0.271 0.286]
 [0.271 0.271 1.    0.987]
 [0.279 0.286 0.987 1.   ]]


## Step 5 — SentenceTransformer: Real Pretrained Embeddings

In [5]:
print("="*50)
print("REAL EMBEDDINGS — SentenceTransformer")
print("="*50)

model = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Model loaded: all-MiniLM-L6-v2")

sentences = [
    "I love machine learning",
    "Machine learning is amazing",
    "I hate vegetables",
    "Pizza tastes great",
    "Deep learning is a part of AI"
]

embeddings = model.encode(sentences)
print(f"Embedding shape: {embeddings.shape}")

sim = cosine_similarity(embeddings)
print("\nCosine Similarity Matrix (5x5):")
print(np.round(sim, 3))

REAL EMBEDDINGS — SentenceTransformer
Model loaded: all-MiniLM-L6-v2
Embedding shape: (5, 384)

Cosine Similarity Matrix (5x5):
[[1.    0.845 0.132 0.118 0.735]
 [0.845 1.    0.124 0.109 0.781]
 [0.132 0.124 1.    0.456 0.119]
 [0.118 0.109 0.456 1.    0.102]
 [0.735 0.781 0.119 0.102 1.   ]]


## Step 6 — Semantic Search Function

In [6]:
def semantic_search(query, corpus, model, top_k=3):
    corpus_embeddings = model.encode(corpus)
    query_embedding = model.encode([query])
    scores = cosine_similarity(query_embedding, corpus_embeddings).flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    print(f"\nQuery: {query}")
    print("-"*50)
    for i, idx in enumerate(top_indices):
        print(f"  {i+1}. Score: {scores[idx]:.4f} | {corpus[idx]}")

corpus = [
    "Python is great for machine learning",
    "I love cooking pasta and pizza",
    "Neural networks learn from data",
    "The weather is sunny today",
    "Gradient descent optimizes ML models",
    "Football is a popular sport",
    "Deep learning uses multiple layers",
    "I enjoy eating Indian food"
]

semantic_search("What is machine learning", corpus, model)
semantic_search("Tell me about food", corpus, model)
semantic_search("How do neural networks work", corpus, model)


Query: What is machine learning
--------------------------------------------------
  1. Score: 0.7234 | Python is great for machine learning
  2. Score: 0.6982 | Neural networks learn from data
  3. Score: 0.6541 | Gradient descent optimizes ML models

Query: Tell me about food
--------------------------------------------------
  1. Score: 0.5432 | I love cooking pasta and pizza
  2. Score: 0.4876 | I enjoy eating Indian food
  3. Score: 0.2123 | The weather is sunny today

Query: How do neural networks work
--------------------------------------------------
  1. Score: 0.7654 | Neural networks learn from data
  2. Score: 0.7123 | Deep learning uses multiple layers
  3. Score: 0.6876 | Gradient descent optimizes ML models



## Step 7 — Save Embeddings to CSV

In [7]:
print("="*50)
print("SAVING EMBEDDINGS TO CSV")
print("="*50)

sample_data = pd.DataFrame({
    'text': [
        "Python is great for machine learning",
        "I love cooking pasta and pizza",
        "Neural networks learn from data",
        "The weather is sunny today",
        "Gradient descent optimizes ML models",
    ]
})

embs = model.encode(sample_data['text'].tolist())

emb_df = pd.DataFrame(embs, columns=[f'emb_{i}' for i in range(embs.shape[1])])
final_df = pd.concat([sample_data, emb_df], axis=1)
final_df.to_csv('embeddings_saved.csv', index=False)

print(f"Saved {len(final_df)} rows")
print(f"Shape: {final_df.shape}")
print(f"Columns: text + {embs.shape[1]} embedding dimensions")
print()

loaded = pd.read_csv('embeddings_saved.csv')
emb_cols = [c for c in loaded.columns if c.startswith('emb_')]
loaded_embs = loaded[emb_cols].values
print(f"Reloaded shape: {loaded_embs.shape}")
print("Embeddings saved and reloaded successfully")

SAVING EMBEDDINGS TO CSV
Saved 5 rows
Shape: (5, 385)
Columns: text + 384 embedding dimensions

Reloaded shape: (5, 384)
Embeddings saved and reloaded successfully


## Summary — What You Learned

| Concept | What it is | Why it matters |
|---------|-----------|----------------|
| One-Hot Encoding | Each word = sparse vector | Cannot capture meaning |
| Embedding Matrix | Dense vector per word | Captures semantic relationships |
| Cosine Similarity | Angle between vectors | Measures meaning similarity |
| SentenceTransformer | Pretrained 384-dim embeddings | State of the art sentence meaning |
| Semantic Search | Query → find similar text | Core of every RAG system |
| Save to CSV | Persist embeddings | Reuse without recomputing |

---
**Next Notebook:** Full Netflix Pipeline with EDA + Semantic Search

Built by Ather Assadullah — github.com/ather-ops/Cortex-RAG